# 05 · Deploy Chat App

Builds a small Streamlit chat UI on top of the deployed `insurance-rag-router` Model Serving
endpoint, and creates + deploys it as a **Databricks App** so it can be shared with users who
don't have workspace/notebook access.

## Setup

Installs the Databricks SDK (for the Apps + Serving Endpoints APIs) and restarts Python.

In [ ]:
%pip install --upgrade databricks-sdk
%restart_python

## Configuration

Points at the already-deployed router serving endpoint (see notebook 04) and names the app +
its workspace source folder. `APP_NAME` must be unique in the workspace and DNS-safe
(lowercase letters, numbers, hyphens).

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

CATALOG = "workspace"
SCHEMA = "insurance"
SERVING_ENDPOINT_NAME = "insurance-rag-router"
APP_NAME = "insurance-rag-chat"

CURRENT_USER = w.current_user.me().user_name
SOURCE_CODE_PATH = f"/Workspace/Users/{CURRENT_USER}/RAG_Project/insurance_rag_app"

print(f"Serving endpoint : {SERVING_ENDPOINT_NAME}")
print(f"App name         : {APP_NAME}")
print(f"Source path      : {SOURCE_CODE_PATH}")

## Confirm the serving endpoint is ready

Model Serving endpoints with `scale_to_zero_enabled` can take a couple of minutes to spin back
up from a cold start. Wait here (rather than fail immediately) so the app isn't wired up
against an endpoint that isn't actually answering yet.

In [ ]:
import time
import pandas as pd

def wait_for_endpoint_ready(name: str, timeout_s: int = 300):
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        ep = w.serving_endpoints.get(name)
        if ep.state and ep.state.ready and ep.state.ready.value == "READY" and ep.state.config_update.value == "NOT_UPDATING":
            print(f"Endpoint '{name}' is READY.")
            return
        print(f"Waiting on endpoint '{name}'... state={ep.state}")
        time.sleep(15)
    raise TimeoutError(f"Endpoint '{name}' did not become READY within {timeout_s}s")

wait_for_endpoint_ready(SERVING_ENDPOINT_NAME)

# Cold-start warm-up + smoke test call (scale-to-zero endpoints need an actual invocation
# to spin compute up; READY state alone doesn't guarantee the container is warm)
test_resp = w.serving_endpoints.query(
    name=SERVING_ENDPOINT_NAME,
    dataframe_records=[{"question": "What is excluded under the Critical Illness policy?"}],
)
print("Smoke test prediction:")
print(test_resp.predictions)

## App source: Streamlit chat UI

A minimal chat interface. It reads `SERVING_ENDPOINT_NAME` from the app's environment (set in
`app.yaml` below) and calls it with `WorkspaceClient()` — inside a deployed Databricks App this
authenticates automatically as the app's own service principal, scoped to only the resources
declared in `app.yaml`. No token is ever handled in application code.

In [ ]:
APP_PY = '''import json
import urllib.parse

import streamlit as st
from databricks.sdk import WorkspaceClient

SERVING_ENDPOINT_NAME = "insurance-rag-router"
MAX_HISTORY_TURNS = 3  # how many prior user/assistant exchanges to carry forward

st.set_page_config(page_title="Insurance Assistant", page_icon="\U0001F4CB")

# --- Watermark: tiled "@DataArchitectStudio" behind the app content ---
_watermark_svg = """<svg xmlns="http://www.w3.org/2000/svg" width="340" height="170">
<text x="10" y="95" font-size="20" fill="rgba(128,128,128,0.16)" transform="rotate(-28 170 85)" font-family="Helvetica, Arial, sans-serif">@DataArchitectStudio</text>
</svg>"""
_watermark_uri = "data:image/svg+xml," + urllib.parse.quote(_watermark_svg)
st.markdown(
    f"""<style>
.stApp {{
    background-image: url("{_watermark_uri}");
    background-repeat: repeat;
}}
</style>""",
    unsafe_allow_html=True,
)

st.title("\U0001F4CB Health Insurance Assistant")
st.caption(
    "Ask about policy coverage, exclusions, waiting periods, or your own premiums/claims. "
    "Answers are routed automatically to policy-document search, structured account data, or both."
)

@st.cache_resource
def get_client():
    return WorkspaceClient()

def build_history_json(history: list) -> str:
    # The model is stateless per call — recent turns travel with every request as
    # structured history_json so it can resolve follow-ups like "give me the details"
    # or "what is the premium for this?" instead of treating them as brand-new questions.
    recent = history[-(MAX_HISTORY_TURNS * 2):]
    # Pair up user/assistant messages into {question, source_used, answer} turns
    pairs = []
    pending_question = None
    for m in recent:
        if m["role"] == "user":
            pending_question = m["content"]
        elif m["role"] == "assistant" and pending_question is not None:
            pairs.append({
                "question": pending_question,
                "source_used": m.get("source_used", "UNKNOWN"),
                "answer": m["content"],
            })
            pending_question = None
    return json.dumps(pairs)

def ask(question: str, history: list) -> dict:
    client = get_client()
    resp = client.serving_endpoints.query(
        name=SERVING_ENDPOINT_NAME,
        dataframe_records=[{"question": question, "history_json": build_history_json(history)}],
    )
    # predict() returns a single dict for one row, or a list of dicts for multiple rows
    preds = resp.predictions
    return preds[0] if isinstance(preds, list) else preds

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

question = st.chat_input("Ask a question about your health insurance policy...")
if question:
    history_before = list(st.session_state.messages)
    st.session_state.messages.append({"role": "user", "content": question})
    with st.chat_message("user"):
        st.markdown(question)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                result = ask(question, history_before)
                answer = result.get("answer", "(no answer returned)")
                source = result.get("source_used", "UNKNOWN")
                st.markdown(answer)
                with st.expander(f"Details · routed to {source}"):
                    if result.get("sql_query"):
                        st.code(result["sql_query"], language="sql")
                    if result.get("vector_chunks_used"):
                        st.write(f"Policy chunks retrieved: {result['vector_chunks_used']}")
                    if result.get("sql_error"):
                        st.warning(f"SQL path error: {result['sql_error']}")
                    if result.get("vector_search_error"):
                        st.warning(f"Vector search error: {result['vector_search_error']}")
            except Exception as e:
                answer = f"Sorry, the assistant hit an error: {e}"
                source = "ERROR"
                st.error(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer, "source_used": source})
'''

REQUIREMENTS_TXT = "streamlit\ndatabricks-sdk\n"

APP_YAML = '''command: ["streamlit", "run", "app.py"]

env:
  - name: "SERVING_ENDPOINT_NAME"
    value: "insurance-rag-router"
'''

print("App source files defined:", "app.py", "requirements.txt", "app.yaml")

## Upload the source files to the workspace

Databricks Apps deploy from a workspace folder, so the three files above are written there
directly via the Workspace API (no local git checkout on the compute needed).

In [ ]:
from databricks.sdk.service.workspace import ImportFormat

w.workspace.mkdirs(SOURCE_CODE_PATH)

def upload_text_file(path: str, content: str):
    w.workspace.upload(path, content.encode("utf-8"), format=ImportFormat.AUTO, overwrite=True)

upload_text_file(f"{SOURCE_CODE_PATH}/app.py", APP_PY)
upload_text_file(f"{SOURCE_CODE_PATH}/requirements.txt", REQUIREMENTS_TXT)
upload_text_file(f"{SOURCE_CODE_PATH}/app.yaml", APP_YAML)

for f in w.workspace.list(SOURCE_CODE_PATH):
    print(f.path)

## Create the app (idempotent) and grant it access to the serving endpoint

Declaring the serving endpoint as an app **resource** with `CAN_QUERY` permission provisions a
scoped OAuth grant for the app's service principal — that's what lets `app.py` call
`WorkspaceClient()` with no credentials of its own.

In [ ]:
from databricks.sdk.service.apps import (
    App,
    AppResource,
    AppResourceServingEndpoint,
    AppResourceServingEndpointServingEndpointPermission,
)
from databricks.sdk.errors import ResourceAlreadyExists

app_def = App(
    name=APP_NAME,
    description="Chat UI for the Insurance RAG Router agent",
    resources=[
        AppResource(
            name="insurance-rag-router",
            serving_endpoint=AppResourceServingEndpoint(
                name=SERVING_ENDPOINT_NAME,
                permission=AppResourceServingEndpointServingEndpointPermission.CAN_QUERY,
            ),
        )
    ],
)

try:
    app = w.apps.create_and_wait(app=app_def)
    print(f"Created app '{APP_NAME}'")
except ResourceAlreadyExists:
    print(f"App '{APP_NAME}' already exists — updating its resource grants instead")
    app = w.apps.update(name=APP_NAME, app=app_def)

print(f"App URL: {app.url}")

## Deploy the source code

Points the app at the workspace folder uploaded above and waits for the deployment to reach
`SUCCEEDED`.

In [ ]:
from databricks.sdk.service.apps import AppDeployment

deployment = w.apps.deploy_and_wait(
    app_name=APP_NAME,
    app_deployment=AppDeployment(source_code_path=SOURCE_CODE_PATH),
)

print(f"Deployment status: {deployment.status.state if deployment.status else 'unknown'}")

final_app = w.apps.get(APP_NAME)
print(f"\nApp is live at: {final_app.url}")
print("Share this URL with anyone granted CAN_USE permission on the app.")

## (Optional) Grant other users access

By default only the creator can open the app. Uncomment and adjust to share it — grants are
scoped to the app, not the underlying serving endpoint or Unity Catalog data.

In [ ]:
# from databricks.sdk.service.apps import AppAccessControlRequest, AppPermissionLevel
#
# w.apps.set_permissions(
#     app_name=APP_NAME,
#     access_control_list=[
#         AppAccessControlRequest(user_name="teammate@example.com", permission_level=AppPermissionLevel.CAN_USE),
#     ],
# )
# print("Granted access.")